# Road Skeletonizer — Multi-city test

Tests Budapest, London, and Milan with identical parameters.

In [2]:
import time
import osmnx as ox
import networkx as nx
import pandas as pd

from road_skeletonizer import RoadSkeletonizer

print("Libraries loaded.")

Libraries loaded.


In [3]:
# --- shared parameters ---
ROAD_TAGS = {
    "highway": ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link"]
}
BUFFER_SIZE       = 100   # metres (Web Mercator)
MIN_LENGTH        = 300   # prune dangles shorter than this
MERGE_DISTANCE    = 250   # merge nodes closer than this

CITIES = {
    "Budapest": "Budapest, Hungary",
    "London":   "London, United Kingdom",
    "Milan":    "Milan, Italy",
}

## Fetch city boundaries

In [4]:
polygons = {}
for city, query in CITIES.items():
    gdf = ox.geocode_to_gdf(query)
    polygons[city] = gdf.geometry.iloc[0]
    bounds = polygons[city].bounds
    print(f"{city:10s}  bounds={bounds}")

Budapest    bounds=(18.9251057, 47.3496899, 19.3349258, 47.6131468)
London      bounds=(-0.5103751, 51.2867601, 0.3340155, 51.6918741)
Milan       bounds=(9.0408867, 45.3867381, 9.2781103, 45.5358482)


## Run skeletonizer for each city

In [5]:
results = {}

for city, polygon in polygons.items():
    print(f"\n{'='*50}")
    print(f"  {city}")
    print(f"{'='*50}")

    rs = RoadSkeletonizer(
        buffer_size=BUFFER_SIZE,
        road_tags=ROAD_TAGS,
        simplify_min_length=MIN_LENGTH,
        simplify_merge_distance=MERGE_DISTANCE,
        verbose=False,
        timing=True,
    )

    t0 = time.perf_counter()
    rs.fit(polygon)
    elapsed = time.perf_counter() - t0

    G = rs.G
    degrees = [G.degree(n) for n in G.nodes()]

    results[city] = {
        "osm_features":    len(rs.roads),
        "skeleton_lines":  len(rs.linestring_skeleton) if rs.linestring_skeleton else 0,
        "nodes":           G.number_of_nodes(),
        "edges":           G.number_of_edges(),
        "connected":       nx.is_connected(G),
        "avg_degree":      round(sum(degrees) / len(degrees), 2) if degrees else None,
        "elapsed_s":       round(elapsed, 1),
        "skeletonizer":    rs,
    }

    print(f"  → {G.number_of_nodes()} nodes, {G.number_of_edges()} edges  ({elapsed:.1f}s total)")


  Budapest


TypeError: 'NoneType' object is not iterable

## Summary table

In [ ]:
summary = pd.DataFrame(
    [
        {
            "city":           city,
            "osm_features":   v["osm_features"],
            "skeleton_lines": v["skeleton_lines"],
            "nodes":          v["nodes"],
            "edges":          v["edges"],
            "connected":      v["connected"],
            "avg_degree":     v["avg_degree"],
            "time_s":         v["elapsed_s"],
        }
        for city, v in results.items()
    ]
).set_index("city")

summary

## Maps

In [ ]:
# Budapest
results["Budapest"]["skeletonizer"].plot_polygon_skeleton_folium(
    plot_highways=True, plot_buffered_shape=False, plot_linestring=False, plot_polygon=True
)

In [ ]:
# London
results["London"]["skeletonizer"].plot_polygon_skeleton_folium(
    plot_highways=True, plot_buffered_shape=False, plot_linestring=False, plot_polygon=True
)

In [ ]:
# Milan
results["Milan"]["skeletonizer"].plot_polygon_skeleton_folium(
    plot_highways=True, plot_buffered_shape=False, plot_linestring=False, plot_polygon=True
)